# SARIMAX Forecasting on AMI Timeseries

This notebook defines a reusable SARIMAX forecasting pipeline for a single SKU:
- Load and filter the AMI timeseries data
- Convert it to a monthly time series
- Train a SARIMAX model
- Perform a backtest on the last months
- Generate a multi-step forecast
- Visualise results with interactive Plotly figures

In [116]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

def load_timeseries(base_dir="..", filename="timeseries.csv"):
    data_path = os.path.join(os.path.abspath(base_dir), "data", filename)
    df = pd.read_csv(data_path)
    df["date"] = pd.to_datetime(df["date"])
    return df

def filter_sku(df, sku_id):
    df_sku = df[df["sku"] == sku_id].copy()
    df_sku = df_sku.sort_values("date")
    return df_sku

def to_monthly_series(df_sku):
    s = df_sku.set_index("date")["qty"].sort_index()
    s = s.asfreq("MS")
    return s

## SARIMAX model training and forecasting

The functions below:
- Configure and fit a SARIMAX model on a given series
- Produce a forecast for a configurable number of steps

In [117]:
def train_sarimax(y_train, order=(1, 0, 1), seasonal_order=(1, 0, 1, 12)):
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(disp=False)
    return res

def forecast_sarimax(fitted_model, steps):
    forecast_res = fitted_model.get_forecast(steps=steps)
    mean_forecast = forecast_res.predicted_mean
    conf_int = forecast_res.conf_int()
    return mean_forecast, conf_int

## SARIMAX model training and forecasting

The functions below:
- Configure and fit a SARIMAX model on a given series
- Produce a forecast for a configurable number of steps

In [118]:
def train_sarimax(y_train, order=(1, 0, 1), seasonal_order=(1, 0, 1, 12)):
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(disp=False)
    return res

def forecast_sarimax(fitted_model, steps):
    forecast_res = fitted_model.get_forecast(steps=steps)
    mean_forecast = forecast_res.predicted_mean
    conf_int = forecast_res.conf_int()
    return mean_forecast, conf_int

## Backtesting

The function below:
- Splits the series into train and test based on a backtest horizon in months
- Fits a SARIMAX model on the train part
- Forecasts the test horizon
- Computes MAE and RMSE
- Returns metrics and a combined DataFrame for plotting

In [119]:
def backtest_sarimax(y, backtest_months=6, order=(1, 0, 1), seasonal_order=(1, 0, 1, 12)):
    y = y.sort_index()
    last_date = y.index.max()
    cutoff = last_date - pd.DateOffset(months=backtest_months)

    y_train = y[y.index <= cutoff]
    y_test = y[y.index > cutoff]

    fitted = train_sarimax(y_train, order=order, seasonal_order=seasonal_order)
    mean_forecast, _ = forecast_sarimax(fitted, steps=len(y_test))

    mean_forecast = mean_forecast.reindex(y_test.index)

    eval_df = pd.DataFrame(
        {
            "date": y_test.index,
            "actual": y_test.values,
            "forecast": mean_forecast.values,
        }
    )

    mae = mean_absolute_error(eval_df["actual"], eval_df["forecast"])
    rmse = float(np.sqrt(mean_squared_error(eval_df["actual"], eval_df["forecast"])))

    return {"mae": float(mae), "rmse": rmse, "eval_df": eval_df, "fitted": fitted}

## Plotly visualisations

The functions below:
- Build an interactive forecast figure (history + future forecast + confidence bands)
- Build an interactive backtest figure (actual vs forecast on the holdout period)

In [120]:
def make_forecast_figure(y, forecast_mean, conf_int, sku_id):
    hist = y.sort_index()
    fc = forecast_mean
    ci = conf_int

    x_band = list(ci.index) + list(ci.index[::-1])
    y_band = list(ci.iloc[:, 0]) + list(ci.iloc[:, 1][::-1])

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=hist.index,
            y=hist.values,
            mode="lines",
            name="History",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=fc.index,
            y=fc.values,
            mode="lines",
            name="Forecast",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=x_band,
            y=y_band,
            fill="toself",
            name="Confidence interval",
            opacity=0.2,
            line=dict(width=0),
            showlegend=True,
        )
    )

    fig.update_layout(
        title=f"SARIMAX forecast – {sku_id}",
        xaxis_title="Date",
        yaxis_title="Quantity",
        legend_title="Legend",
    )
    return fig

def make_backtest_figure(eval_df, sku_id):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=eval_df["date"],
            y=eval_df["actual"],
            mode="lines+markers",
            name="Actual",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=eval_df["date"],
            y=eval_df["forecast"],
            mode="lines+markers",
            name="Forecast",
        )
    )

    fig.update_layout(
        title=f"SARIMAX backtest – {sku_id}",
        xaxis_title="Date",
        yaxis_title="Quantity",
        legend_title="Legend",
    )
    return fig

## End-to-end SARIMAX pipeline

The function below:

- Loads the timeseries from `../data/timeseries.csv`
- Filters to a selected SKU
- Converts it to a monthly series
- Performs a backtest on the last months
- Re-fits on the full series
- Produces a multi-step forecast
- Returns metrics and Plotly figures

In [121]:
def run_full_sarimax_pipeline(
    base_dir="..",
    filename="timeseries.csv",
    sku_id="SKU1_001",
    forecast_months=12,
    backtest_months=6,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 12),
):
    df = load_timeseries(base_dir=base_dir, filename=filename)
    df_sku = filter_sku(df, sku_id)
    y = to_monthly_series(df_sku)

    bt = backtest_sarimax(y, backtest_months=backtest_months, order=order, seasonal_order=seasonal_order)
    eval_df = bt["eval_df"]
    backtest_metrics = {"mae": bt["mae"], "rmse": bt["rmse"]}

    full_fitted = train_sarimax(y, order=order, seasonal_order=seasonal_order)
    mean_forecast, conf_int = forecast_sarimax(full_fitted, steps=forecast_months)

    forecast_fig = make_forecast_figure(y, mean_forecast, conf_int, sku_id)
    backtest_fig = make_backtest_figure(eval_df, sku_id)

    result = {
        "sku_id": sku_id,
        "series": y,
        "full_fitted": full_fitted,
        "forecast_mean": mean_forecast,
        "forecast_conf_int": conf_int,
        "forecast_fig": forecast_fig,
        "backtest_metrics": backtest_metrics,
        "backtest_eval_df": eval_df,
        "backtest_fig": backtest_fig,
    }
    return result

## Get the results

In [122]:
def run_and_display_for_skus(
    skus,
    base_dir="..",
    filename="timeseries.csv",
    forecast_months=12,
    backtest_months=6,
):
    results = {}
    metrics_rows = []
    artifacts_dir = os.path.join(os.path.abspath(base_dir), "artifacts")
    os.makedirs(artifacts_dir, exist_ok=True)
    for sku_id in skus:
        res = run_full_sarimax_pipeline(
            base_dir=base_dir,
            filename=filename,
            sku_id=sku_id,
            forecast_months=forecast_months,
            backtest_months=backtest_months,
        )
        results[sku_id] = res
        m = res["backtest_metrics"]
        metrics_rows.append({"sku_id": sku_id, "mae": m["mae"], "rmse": m["rmse"]})
        print(f"{sku_id} backtest metrics:", m)
        res["forecast_fig"].show()
        res["backtest_fig"].show()
        fc = res["forecast_mean"]
        ci = res["forecast_conf_int"]
        df_fc = pd.DataFrame(
            {
                "date": fc.index,
                "forecast": fc.values,
                "lower": ci.iloc[:, 0].values,
                "upper": ci.iloc[:, 1].values,
            }
        )
        df_fc.to_csv(
            os.path.join(artifacts_dir, f"sarimax_forecast_{sku_id}.csv"),
            index=False,
        )
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(
        os.path.join(artifacts_dir, "sarimax_backtest_metrics.csv"),
        index=False,
    )
    return results

In [123]:
skus = ["SKU1_001", "SKU1_002", "SKU1_003", "SKU1_004", "SKU1_005", "SKU1_006"]
sarimax_results = run_and_display_for_skus(skus)

SKU1_001 backtest metrics: {'mae': 2.892923439881523, 'rmse': 3.5568909344433646}


/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.

/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.



/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.

/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.



SKU1_002 backtest metrics: {'mae': 2.066439211440055, 'rmse': 2.609548736738272}


SKU1_003 backtest metrics: {'mae': 2.794503366361718, 'rmse': 3.49819000143275}


...

SKU1_004 backtest metrics: {'mae': 6.318031739881498, 'rmse': 9.13749607461012}


/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.

/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.



SKU1_005 backtest metrics: {'mae': 1.601554796876081, 'rmse': 1.7473070439776708}


/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.

/Users/kornel/ADAI-individual-repository-2025/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning:

Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.



SKU1_006 backtest metrics: {'mae': 1.3817254369899896, 'rmse': 1.660545369276779}
